# MDN Web Reference — Semantic Vector Index

Build a local RAG-style vector index of the entire MDN web reference
(HTML, CSS, JavaScript, SVG, Accessibility, HTTP, Web APIs)
using **zvec** (in-process vector database) and **sentence-transformers**.

---
### What this does
1. Clones `mdn/content` (depth=1, ~500 MB)
2. Chunks every `.md` page under `files/en-us/web/` by heading
3. Embeds each chunk with `all-MiniLM-L6-v2` (384-dim)
4. Stores vectors + metadata in a zvec collection on disk
5. Lets you query semantically — *"what attributes does <dialog> support?"*
   → returns the relevant MDN page chunks ranked by relevance

In [ ]:
# ── Dependencies ────────────────────────────────────────────────
# Install with uv (recommended) or pip:
#   uv pip install zvec sentence-transformers
import os, re, subprocess, hashlib, sys, json
from pathlib import Path

import zvec
from sentence_transformers import SentenceTransformer

In [ ]:
# ── Configuration ───────────────────────────────────────────────
SKILL_DIR = Path.home() / ".config" / "opencode" / "skills" / "semantic-html"
INDEX_PATH = str(SKILL_DIR / "mdn_index.zvec")
REPO_URL = "https://github.com/mdn/content.git"
REPO_DIR = SKILL_DIR / "mdn_content"
EMBED_MODEL = "all-MiniLM-L6-v2"      # 384-dim, lightweight
EMBED_DIM = 384
COLLECTION_NAME = "mdn_web_reference"

SKILL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── 1. Clone MDN content ────────────────────────────────────────
if not REPO_DIR.exists():
    print("Cloning MDN content (depth=1)... this is ~500 MB")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True, capture_output=True
    )
else:
    print("MDN content already cloned.")

In [ ]:
# ── 2. Chunk markdown pages ─────────────────────────────────────
def chunk_markdown(text: str, min_chars=200, max_chars=1500):
    """Split an MDN page into chunks at heading boundaries."""
    text = re.sub(r'^---\n.*?\n---\n', '', text, flags=re.DOTALL)
    if len(text.strip()) < min_chars:
        return []
    parts = re.split(r'(?=^#{1,3}\s)', text, flags=re.MULTILINE)
    chunks, buffer = [], ""
    for part in parts:
        if not part.strip(): continue
        if len(buffer) + len(part) > max_chars and buffer:
            chunks.append(buffer.strip())
            buffer = part
        else:
            buffer += "\n" + part
    if buffer.strip():
        chunks.append(buffer.strip())
    # Split oversized chunks by paragraph
    final = []
    for c in chunks:
        if len(c) > max_chars:
            buf = ""
            for p in c.split("\n\n"):
                if len(buf) + len(p) > max_chars and buf:
                    final.append(buf.strip())
                    buf = p
                else:
                    buf += "\n\n" + p
            if buf.strip(): final.append(buf.strip())
        else:
            final.append(c.strip())
    return [c for c in final if len(c) >= min_chars]

In [ ]:
# ── 3. Create zvec collection & load embedding model ────────────
schema = zvec.CollectionSchema(
    name=COLLECTION_NAME,
    fields=[
        zvec.FieldSchema("source", zvec.DataType.STRING),
        zvec.FieldSchema("path", zvec.DataType.STRING),
        zvec.FieldSchema("title", zvec.DataType.STRING),
        zvec.FieldSchema("chunk", zvec.DataType.INT32),
    ],
    vectors=zvec.VectorSchema("embedding", zvec.DataType.VECTOR_FP32, EMBED_DIM),
)

collection = zvec.create_and_open(path=INDEX_PATH, schema=schema)
model = SentenceTransformer(EMBED_MODEL)

In [ ]:
# ── 4. Embed and index every page ───────────────────────────────
base_path = REPO_DIR / "files" / "en-us"

# Target these key sections of the MDN reference
sections = [
    "web/html", "web/css", "web/javascript", "web/svg",
    "web/api", "web/accessibility", "web/performance",
    "web/guide", "web/http", "web/security",
    "Learn_web_development/Core/Structuring_content",
]

doc_batch = []
BATCH_SIZE = 64
total = 0

def flush():
    global doc_batch, total
    if not doc_batch: return
    collection.insert(doc_batch)
    total += len(doc_batch)
    print(f"  Indexed {total} chunks...", end="\r")
    doc_batch = []

# ── Phase 1: walk files, collect chunks ───────────────────────
pending = []  # (text, rel_path, title, chunk_index)
for section in sections:
    target = base_path / section
    if not target.exists(): continue
    for root, _, files in os.walk(target):
        for fn in files:
            if not fn.endswith(".md"): continue
            full = Path(root) / fn
            rel = str(full.relative_to(base_path))
            try:
                content = full.read_text(encoding="utf-8")
            except: continue
            title_m = re.search(r'^#{1,3}\s+(.+)$', content, re.MULTILINE)
            title = title_m.group(1).strip() if title_m else rel
            for i, chunk in enumerate(chunk_markdown(content)):
                pending.append((chunk, rel, title, i))

print(f"Found {len(pending)} chunks across {len(sections)} sections")

# ── Phase 2: embed in batches, insert in batches ───────────────
EMBED_BATCH = 32
for start in range(0, len(pending), EMBED_BATCH):
    batch = pending[start:start+EMBED_BATCH]
    texts = [b[0] for b in batch]
    embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    for (text, rel, title, ci), emb in zip(batch, embeddings):
        doc_id = "mdn" + hashlib.sha256(f"{rel}\0{ci}".encode()).hexdigest()[:30]
        doc_batch.append(zvec.Doc(
            id=doc_id,
            vectors={"embedding": emb.tolist()},
            fields={"source": "mdn", "path": rel, "title": title, "chunk": ci},
        ))
        if len(doc_batch) >= BATCH_SIZE: flush()
flush()
collection.commit()
print(f"\n✓ Done. Indexed {total} chunks.")

In [ ]:
# ── 5. Query the index ──────────────────────────────────────────
collection = zvec.open(INDEX_PATH)

query = "what attributes does the <dialog> element support"
emb = model.encode(query, normalize_embeddings=True).tolist()

results = collection.query(
    vectors=zvec.VectorQuery("embedding", vector=emb),
    topk=8,
    output_fields=["path", "title"],
)

for r in results:
    print(f"  [{r.score:.3f}] {r.field('title')}")
    print(f"         {r.field('path')}\n")

In [ ]:
# ── 6. Use reranker for higher precision ────────────────────────
# zvec supports cross-encoder reranking for relevance boost
# Reference: https://zvec.org/en/docs/ai/reranker/

reranker = zvec.ReRanker(model_name="BAAI/bge-reranker-v2-m3")

results = collection.query(
    vectors=zvec.VectorQuery("embedding", vector=emb),
    topk=20,          # retrieve more candidates
    reranker=reranker, # rerank for precision
    output_fields=["path", "title"],
)

for r in results[:8]:
    print(f"  [{r.score:.3f}] {r.field('title')}")
    print(f"         {r.field('path')}\n")